In [1]:
# 导入基础库
import json
import sys
from pathlib import Path
from prompts import fill_prompt
import re

# 将当前目录加入系统路径，确保可以导入自定义模块
sys.path.append(str(Path.cwd()))

# 导入自定义模块
from config import OLLAMA_CHAT_MODEL, OLLAMA_BASE_URL
from utils import ollama_chat
from prompts import DECOMPOSE_PROMPT_HEAD

print(f"当前使用的 Ollama 模型：{OLLAMA_CHAT_MODEL}")
print(f"Ollama 服务地址：{OLLAMA_BASE_URL}")

当前使用的 Ollama 模型：qwen3.5:9b
Ollama 服务地址：http://localhost:11434


In [2]:
# 单元格 2：定义问题分解函数（返回子问题列表，不包含根节点）
def decompose_question(question, max_depth=5, current_depth=0):
    """
    递归地将问题分解为子问题，构建思维导图子树。
    返回一个列表，每个元素为 {"sub_question": str, "state": "Continue"/"End", "depth": int, "children": list}
    """
    if current_depth >= max_depth:
        return [{"sub_question": question, "state": "End", "depth": current_depth, "children": []}]

    prompt = fill_prompt(DECOMPOSE_PROMPT_HEAD, question=question)

    print(f"\n[深度 {current_depth}] 正在分解问题：{question[:50]}...")

    response = ollama_chat(prompt, temperature=0.1, format_json=False)

    print(f"[调试] 原始响应长度：{len(response)} 字符")
    print(f"[调试] 原始响应内容：{repr(response)}")

    array_match = re.search(r'\[.*\]', response, re.DOTALL)
    if array_match:
        json_str = array_match.group(0)
        try:
            subqs = json.loads(json_str)
            if isinstance(subqs, list):
                valid_subqs = []
                for item in subqs:
                    if isinstance(item, dict) and "sub_question" in item and "state" in item:
                        valid_subqs.append(item)
                if valid_subqs:
                    subqs = valid_subqs
                else:
                    raise ValueError("数组中无有效子问题")
            else:
                raise ValueError("提取的内容不是列表")
        except Exception as e:
            print(f"解析数组失败：{e}")
            subqs = None
    else:
        obj_match = re.search(r'\{.*\}', response, re.DOTALL)
        if obj_match:
            try:
                obj = json.loads(obj_match.group(0))
                if isinstance(obj, dict) and "sub_question" in obj:
                    print("[警告] 模型返回单个对象，已自动包装为列表")
                    subqs = [obj]
                else:
                    raise ValueError("对象缺少 sub_question 字段")
            except Exception as e:
                print(f"解析对象失败：{e}")
                subqs = None
        else:
            print("未找到任何 JSON 结构")
            subqs = None

    if subqs is None:
        print("回退：将该问题标记为不可分解")
        return [{"sub_question": question, "state": "End", "depth": current_depth, "children": []}]

    result_nodes = []
    for item in subqs:
        sub_q = item.get("sub_question", "")
        state = item.get("state", "End")
        node = {
            "sub_question": sub_q,
            "state": state,
            "depth": current_depth,
            "children": []
        }
        if state == "Continue":
            node["children"] = decompose_question(sub_q, max_depth, current_depth + 1)
        else:
            node["children"] = []
        result_nodes.append(node)

    return result_nodes

In [3]:
# 单元格 3：定义树结构展示与叶子收集函数
def print_mind_map(nodes, indent=0):
    """递归打印思维导图树结构"""
    for node in nodes:
        prefix = "  " * indent + "├─ "
        print(f"{prefix}Q: {node['sub_question']} [状态: {node['state']}, 深度: {node['depth']}]")
        if node.get("children"):
            print_mind_map(node["children"], indent + 1)

def collect_leaf_questions(nodes):
    """收集所有叶子节点的子问题文本"""
    leafs = []
    for node in nodes:
        if not node.get("children"):
            leafs.append(node["sub_question"])
        else:
            leafs.extend(collect_leaf_questions(node["children"]))
    return leafs

In [4]:
# 单元格 4：构建包含根节点的完整思维导图
def build_full_mind_map(original_question, max_depth=3):
    """
    构建以原始问题为根节点的完整思维导图树。
    返回一个列表，其中只包含一个根节点（方便与现有接口兼容）。
    """
    root = {
        "sub_question": original_question,
        "state": "Continue",   # 根问题需要分解
        "depth": 0,
        "children": []
    }
    # 对根问题进行分解，子问题作为其 children
    root["children"] = decompose_question(original_question, max_depth, current_depth=1)
    return [root]

In [5]:
# 单元格 5：测试复杂问题并生成完整树
complex_question = "陈氏太极拳的动作要素有哪些？陈氏太极拳的代表传承人有哪些？"

print("===== 构建完整思维导图（含根节点） =====")
full_tree = build_full_mind_map(complex_question, max_depth=3)

print("\n===== 完整思维导图结构 =====")
print_mind_map(full_tree)

print("\n===== 叶子节点问题列表 =====")
leaf_questions = collect_leaf_questions(full_tree)
for i, q in enumerate(leaf_questions, 1):
    print(f"{i}. {q}")

===== 构建完整思维导图（含根节点） =====

[深度 1] 正在分解问题：陈氏太极拳的动作要素有哪些？陈氏太极拳的代表传承人有哪些？...
[调试] 原始响应长度：105 字符
[调试] 原始响应内容：'[{"sub_question": "陈氏太极拳的动作要素有哪些？", "state": "End"}, {"sub_question": "陈氏太极拳的代表传承人有哪些？", "state": "End"}]'

===== 完整思维导图结构 =====
├─ Q: 陈氏太极拳的动作要素有哪些？陈氏太极拳的代表传承人有哪些？ [状态: Continue, 深度: 0]
  ├─ Q: 陈氏太极拳的动作要素有哪些？ [状态: End, 深度: 1]
  ├─ Q: 陈氏太极拳的代表传承人有哪些？ [状态: End, 深度: 1]

===== 叶子节点问题列表 =====
1. 陈氏太极拳的动作要素有哪些？
2. 陈氏太极拳的代表传承人有哪些？


In [6]:
# 单元格 6：保存思维导图到文件
output_file = "data/mind_map_sample.json"
Path("data").mkdir(exist_ok=True)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(full_tree, f, ensure_ascii=False, indent=2)

print(f"思维导图已保存至：{output_file}")

思维导图已保存至：data/mind_map_sample.json
